# 09 — Three-body solver baseline

This notebook validates the reference 3D three-body solver and saves a figure-eight trajectory.

It uses the known equal-mass figure-eight initial condition embedded in the z=0 plane,
then solves the Newtonian equations with `solve_ivp/DOP853` and a fixed-step RK4 solver.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "three_body").exists():
            return p
    raise RuntimeError("Start Jupyter from the repository root.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from three_body import ThreeBodyConfig, ThreeBodySystem3D, figure_eight_state_3d
from three_body.data import generate_reference_figure_eight, save_trajectory_dataset
from three_body.system import integrate_rk4_fixed
from three_body.eval import rmse_per_step
from three_body.viz import plot_3d_trajectories, animate_3d_comparison, save_gif

DATA_DIR = REPO_ROOT / "data" / "three_body"
RESULTS_DIR = REPO_ROOT / "results" / "three_body"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT)

In [ ]:
cfg = ThreeBodyConfig(
    gravity=1.0,
    masses=(1.0, 1.0, 1.0),
    softening=0.0,
    damping=0.0,
)
system = ThreeBodySystem3D(cfg)

DT = 0.005
STEPS = 1400
x0 = figure_eight_state_3d()

traj_solve_ivp, t = generate_reference_figure_eight(
    system, x0, steps=STEPS, dt=DT, solver="solve_ivp"
)
traj_solve_ivp = traj_solve_ivp[0]
traj_rk4 = integrate_rk4_fixed(system, x0, steps=STEPS, dt=DT)

print("solve_ivp:", traj_solve_ivp.shape)
print("rk4:", traj_rk4.shape)
print("energy start/end solve_ivp:", system.energy(traj_solve_ivp[[0, -1]]))
print("energy start/end rk4:", system.energy(traj_rk4[[0, -1]]))

In [ ]:
err_rk4 = rmse_per_step(system, traj_solve_ivp, traj_rk4)

plt.figure(figsize=(8, 4))
plt.plot(t, err_rk4)
plt.xlabel("time")
plt.ylabel("RMSE vs solve_ivp")
plt.title("Fixed RK4 vs DOP853 reference")
plt.grid(True, alpha=0.3)
plt.show()

fig, ax = plot_3d_trajectories(traj_solve_ivp, traj_rk4, title="Figure-eight: solve_ivp vs fixed RK4")
fig.savefig(RESULTS_DIR / "figure8_solve_ivp_vs_rk4_paths.png", dpi=160)
plt.show()

In [ ]:
metadata = {
    "system": system.metadata(),
    "dt": DT,
    "steps": STEPS,
    "solver": "solve_ivp_DOP853",
}
out = DATA_DIR / "figure8_reference_solve_ivp.npz"
save_trajectory_dataset(out, traj_solve_ivp[None], t, metadata=metadata)
print("saved:", out)

anim = animate_3d_comparison(traj_solve_ivp, traj_rk4, interval_ms=20, title="Figure-eight: solve_ivp vs RK4", trail=180)
gif_path = RESULTS_DIR / "figure8_solve_ivp_vs_rk4.gif"
save_gif(anim, gif_path, fps=30)
print("saved:", gif_path)